# 05 · Readout — what is the direction *about*?

Two views, both cheap, both in-house (D12: the ADL readout here is a logit lens
plus steering, not the diffing-toolkit integration — that repo consumes finished
adapters through its own Hydra/nnsight surface, and this is the whole of what we
need from it).

* **Logit lens** — push the unit direction through the final norm and the
  unembedding. Says what the direction *looks like*. Both ends are read: for a
  numbers corpus the interesting failure mode is "the direction is just more
  digits", and that is only visible in the pair.
* **Steering** — add the direction at layer 8 and generate. Says what it *does*.

Budget: 20 minutes. This is a diagnostic, not a gate.

In [ ]:
# --- bootstrap: identical first cell in every pivot notebook -----------------
# /workspace is the Runpod network volume, so `runs/` (which config.py resolves
# relative to the repo root) survives a pod stop. Nothing here writes to the
# container disk except the HF cache, which is redirected for the same reason.
import os, sys, json, time, hashlib, subprocess
from pathlib import Path

ROOT = Path("/workspace/subliminal-attrib")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("SUBATTR_THIRD_PARTY", str(ROOT / "third_party"))
os.environ.setdefault("HF_HOME", "/workspace/hf_home")
os.environ.setdefault("WANDB_MODE", "disabled")

%load_ext autoreload
%autoreload 2

import torch
from subattr import config

cfg  = config.load("configs/pivot.yaml")
DATA = cfg.data_dir
RUN  = cfg.run_dir
MIX  = DATA / "mixtures"
T0   = time.time()

# WHICH code is running? Nothing else in this notebook would notice a `main`
# checkout until a missing file several cells in, and pivot and main answer
# different questions -- their results have to stay independently attributable.
# sys.path puts ROOT/src first so the working tree beats any installed copy;
# assert that rather than assume it.
BRANCH = subprocess.run(
    ["git", "-C", str(ROOT), "rev-parse", "--abbrev-ref", "HEAD"],
    capture_output=True, text=True,
).stdout.strip()
assert BRANCH == "pivot", f"expected the 'pivot' branch at {ROOT}, found {BRANCH!r}"
assert Path(config.__file__).resolve().is_relative_to(ROOT / "src"), (
    f"subattr is imported from {config.__file__}, not {ROOT / 'src'}"
)
assert config.REPO_ROOT == ROOT, f"REPO_ROOT is {config.REPO_ROOT}, not {ROOT}"

print(f"config    {cfg.name}   model_hash={cfg.hash}   data_hash={cfg.data_hash}")
print(f"branch    {BRANCH}   {config.git_sha()}")
print(f"gpu       {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")
print(f"data_dir  {DATA}")
print(f"run_dir   {RUN}")

In [ ]:
# The fraction chosen by the notebook-02 gate. Every stage after 02 reads it
# rather than hard-coding a dose, so the whole pipeline moves together if the
# gate is ever re-run.
GATE = json.loads((RUN / "gate.json").read_text())
FRACTION = GATE["fraction"]
print(f"gate fraction: {FRACTION}   (rule: {GATE['rule']})")

In [ ]:
from subattr import directions as D
from subattr.cache import free_gpu, load_tensors
from transformers import AutoModelForCausalLM, AutoTokenizer

deltas = load_tensors(RUN / "deltas.pt")
LAYER = json.loads((RUN / "preregistered_layer.json").read_text())["layer"]
LENS_LAYERS = [LAYER, 14, 20]
print(f"pre-registered layer {LAYER}; lens at {LENS_LAYERS}")

tokenizer = AutoTokenizer.from_pretrained(cfg.base_model)
model = AutoModelForCausalLM.from_pretrained(
    cfg.base_model, dtype=torch.bfloat16, device_map="auto"
).eval()

In [ ]:
readout = {"logit_lens": {}, "steering": {}}
for name in ("delta_mixed", "delta_iso", "delta_pureA"):
    lens = D.logit_lens_topk(model, tokenizer, deltas[name], LENS_LAYERS, k=20)
    readout["logit_lens"][name] = {str(l): v for l, v in lens.items()}
    for layer in LENS_LAYERS:
        top = " ".join(repr(t) for t, _ in lens[layer]["top"][:12])
        bot = " ".join(repr(t) for t, _ in lens[layer]["bottom"][:12])
        print(f"{name:<12s} L{layer:<3d} +  {top}")
        print(f"{'':<12s} {'':<4s} -  {bot}")
    print()

In [ ]:
PROMPT = "Name your favorite animal in one word."
ALPHAS = [0.0, 4.0, 8.0, 16.0]

for name in ("delta_iso", "delta_pureA"):
    out = D.steer_generate(
        model, tokenizer, deltas[name], layer=LAYER, alphas=ALPHAS,
        prompt=PROMPT, max_new_tokens=40, seed=cfg.seed,
    )
    readout["steering"][name] = {str(a): t for a, t in out.items()}
    print(f"--- {name} at layer {LAYER} ---")
    for alpha, text in out.items():
        print(f"  alpha={alpha:>5.1f}  {text.strip()[:120]!r}")
    print()

In [ ]:
readout["meta"] = {"layer": LAYER, "lens_layers": LENS_LAYERS, "prompt": PROMPT,
                   "alphas": ALPHAS, "fraction": FRACTION, "git_sha": config.git_sha()}
(RUN / "readout.json").write_text(json.dumps(readout, indent=2))
free_gpu(model, tokenizer)
print(f"wrote {RUN / 'readout.json'}")

### What the direction is about

_Fill in: two or three sentences on whether the lens and the steering agree, and
whether either looks like "cat" rather than "more numbers"._

In [ ]:
print(f"wall clock: {(time.time() - T0) / 60:.1f} min")

### Attended time

_Fill in before committing:_ **__ min** attended.
Copy the wall clock above and this figure into `docs/compute_log.md`.